# DSA 504 — Class 7
## Data Cleaning II: Strings, Regular Expressions, and Dates

**Date:** Wednesday, Sep 23
**Reading:** *Practical Business Analytics Using R and Python*, ch. 5 (optional)
**Today:** HW1 is due before class. HW2 is assigned at the end of class.

---

Class 6 fixed missing values, duplicates, and converted `date` to a real datetime column. One problem has been sitting untouched since Class 4: the `category` column has 13 different text values representing only 5 real categories (`"Electronics"`, `"electronics"`, `"ELECTRONICS"`, and so on). **Today we fix that for good.**

### Learning goals
By the end of this class, you will be able to:
- Use pandas' `.str` accessor to clean and standardize text columns
- Write and apply basic regular expressions (regex) to match and clean messy text patterns
- Use `.str.replace()`, `.str.extract()`, and `.str.contains()` with regex
- Fully standardize the `category` column and verify the fix
- Do more advanced work with the `date` column: extracting components and formatting dates for display


In [1]:
import pandas as pd
import numpy as np
import re

sales = pd.read_csv("retail_sales.csv")
sales["date"] = pd.to_datetime(sales["date"])
sales = sales.drop_duplicates()
print(sales.shape)
sales.head()


(4014, 5)


,date,store,category,units_sold,revenue
0,2025-04-22,Rome,Toys,25.0,402.50
1,2025-04-12,Syracuse,electronics,38.0,3154.00
2,2025-12-21,Utica,home goods,42.0,1299.06
3,2025-02-11,Syracuse,Electronics,60.0,5114.40
4,2025-04-10,Syracuse,toys,10.0,147.10


**Note:** the cell above re-does the Class 6 cleaning (dates, duplicates) so this notebook works standalone. The missing-value handling from Class 6 is left for you to reapply in today's guided practice, since the right strategy depends on what you're about to do with the data.


## 1. The `.str` Accessor

pandas gives every text (string) column a special `.str` accessor, which lets you apply string operations to an *entire column at once* instead of looping over it row by row. This should feel familiar — it's the same idea as the whole-column math you did back in Class 5, just for text instead of numbers.


In [2]:
# Without .str, you'd need a loop to lowercase every value -- with it, one line does the whole column
print(sales["category"].str.lower().head(10))


0           toys
1    electronics
2     home goods
3    electronics
4           toys
5    electronics
6      groceries
7     home goods
8      groceries
9      groceries
Name: category, dtype: object


In [3]:
# A few more common .str methods
print(sales["category"].str.upper().head(3))
print(sales["category"].str.strip().head(3))     # removes leading/trailing whitespace
print(sales["category"].str.len().head(3))        # length of each string


0           TOYS
1    ELECTRONICS
2     HOME GOODS
Name: category, dtype: object
0           Toys
1    electronics
2     home goods
Name: category, dtype: object
0     4
1    11
2    10
Name: category, dtype: int64


### Checking for a pattern with `.str.contains()`


In [9]:
# Which rows have "elect" anywhere in the category text (case-insensitive)?
electronics_ish = sales[sales["category"].str.contains("elect", case=False)]
print(electronics_ish["category"].unique())


['electronics' 'Electronics' 'ELECTRONICS']


## 2. Looking at the Full Mess

Before fixing anything, let's see exactly what we're dealing with.


In [10]:
print(sorted(sales["category"].unique()))
print(f"\nTotal distinct text values: {sales['category'].nunique()}")


['Cloth ing', 'Clothing', 'ELECTRONICS', 'Electronics', 'Groceries', 'Home Goods', 'Home_Goods', 'Toys', 'clothing', 'electronics', 'groceries', 'home goods', 'toys']

Total distinct text values: 13


Notice a few different *kinds* of problems mixed together:
- **Capitalization differences**: `"Electronics"` vs `"electronics"` vs `"ELECTRONICS"`
- **Spacing/underscore differences**: `"Home Goods"` vs `"home goods"` vs `"Home_Goods"`
- **An actual typo**: `"Cloth ing"` (a stray space inside the word)

A single fix (like just lowercasing) won't solve all of these — we need a few different tools.


## 3. Step 1 — Fix Case and Whitespace

Lowercasing and stripping whitespace is always a safe first move, and it knocks out a lot of the problem immediately.


In [11]:
sales["category_clean"] = sales["category"].str.lower().str.strip()
print(sorted(sales["category_clean"].unique()))
print(f"\nDistinct values remaining: {sales['category_clean'].nunique()}")


['cloth ing', 'clothing', 'electronics', 'groceries', 'home goods', 'home_goods', 'toys']

Distinct values remaining: 7


That took us from 13 down to fewer distinct values — but check the output above. `"home_goods"` (underscore) is still separate from `"home goods"` (space), and `"cloth ing"` (stray space) is still separate from `"clothing"`. Lowercasing alone doesn't fix structural differences like that.


## 4. Regular Expressions (regex) — Matching Text Patterns

A **regular expression** is a mini pattern-matching language for text. Instead of matching one exact string, a regex can match a whole *family* of similar strings at once. This is the right tool for "any amount of whitespace" or "underscore OR space" style problems.

We'll only need a handful of regex building blocks for this course:

| Pattern | Matches |
|---|---|
| `\s+` | one or more whitespace characters (spaces, tabs) |
| `_` | a literal underscore |
| `[_\s]+` | one or more underscores OR whitespace characters, in any mix |
| `^` | the start of the string |
| `$` | the end of the string |


In [15]:
# A plain string search -- only matches EXACTLY this
sample = "home_goods"
print(re.search("home goods", sample))   # None -- no match, because of the underscore
print(re.search("home.goods", sample))   # matches! "." means "any single character"


None
<re.Match object; span=(0, 10), match='home_goods'>


In [19]:
# re.sub() replaces every match of a pattern with something else
text = "home_goods"
fixed = re.sub(r"[_\s]+", " ", text)   # replace any run of underscores/spaces with a single space
print(fixed)

text2 = "cloth ing"
fixed2 = re.sub(r"[_\s]+", " ", text2)
print(fixed2)   # doesn't fully fix this one -- still "cloth ing", just normalized spacing


home goods
cloth ing


**Important realization:** regex can normalize *separators* (turning `_` and extra spaces into one consistent space), but it can't know that `"cloth ing"` is a typo for `"clothing"` — that requires knowing the *correct* category names in advance, not just pattern-matching. This is a common real-world lesson: cleaning text often needs both a general rule (regex) **and** a specific list of known corrections.


## 5. Applying regex Across a Whole Column with `.str.replace()`

Just like `re.sub()` works on one string, `.str.replace()` with `regex=True` applies the same substitution to an entire column at once.


In [20]:
# Step 2: normalize underscores/extra spaces into single spaces, across the whole column
sales["category_clean"] = sales["category_clean"].str.replace(r"[_\s]+", " ", regex=True)
print(sorted(sales["category_clean"].unique()))


['cloth ing', 'clothing', 'electronics', 'groceries', 'home goods', 'toys']


### Step 3 — Fix the remaining known typo with an explicit mapping

For the one case regex can't solve on its own (`"cloth ing"` → `"clothing"`), we use a plain dictionary lookup with `.replace()` (no regex needed here — this is an exact-value replacement, not a pattern).


In [21]:
category_corrections = {
    "cloth ing": "clothing"
}

sales["category_clean"] = sales["category_clean"].replace(category_corrections)
print(sorted(sales["category_clean"].unique()))
print(f"\nDistinct values remaining: {sales['category_clean'].nunique()}")


['clothing', 'electronics', 'groceries', 'home goods', 'toys']

Distinct values remaining: 5


**We're down to exactly 5 — the real number of categories.** This is the payoff of everything since Class 4: `.groupby("category_clean")` will now give correct, trustworthy results, unlike every grouped example back in Class 5.


In [22]:
# Confirming the fix actually changes the analysis
print("BEFORE cleaning (Class 5's version):")
print(sales.groupby("category")["revenue"].sum().round(2))

print("\nAFTER cleaning:")
print(sales.groupby("category_clean")["revenue"].sum().round(2))


BEFORE cleaning (Class 5's version):
category
Cloth ing      407549.91
Clothing       380245.49
ELECTRONICS    870441.76
Electronics    958509.82
Groceries      359244.48
Home Goods     313182.72
Home_Goods     314445.22
Toys           145971.69
clothing       461761.97
electronics    984799.03
groceries      334076.93
home goods     323621.80
toys           158276.52
Name: revenue, dtype: float64

AFTER cleaning:
category_clean
clothing       1249557.37
electronics    2813750.61
groceries       693321.41
home goods      951249.74
toys            304248.21
Name: revenue, dtype: float64


## 6. Extracting Parts of a String with `.str.extract()`

Sometimes you don't want to replace text — you want to *pull a piece out* of it. `.str.extract()` uses a regex with a "capture group" (parentheses) to grab just the part you want.


In [ ]:
# Example: pulling the year out of a store code like "UTC-2025"
store_codes = pd.Series(["UTC-2025", "ALB-2024", "ROM-2025"])

years = store_codes.str.extract(r"-(\d{4})")   # \d{4} means "exactly 4 digits", ?P<date> to give a column name to the extracted value
print(years)


      0
0  2025
1  2024
2  2025


## 7. More Work with Dates

Class 6 converted `date` to a real datetime column. Now that it's a proper date type, here's more of what becomes possible.


In [29]:
# Extracting different parts of a date
sales["year"] = sales["date"].dt.year
sales["month_name"] = sales["date"].dt.month_name()
sales["quarter"] = sales["date"].dt.quarter

print(sales[["date", "year", "month_name", "quarter"]].head())


        date  year month_name  quarter
0 2025-04-22  2025      April        2
1 2025-04-12  2025      April        2
2 2025-12-21  2025   December        4
3 2025-02-11  2025   February        1
4 2025-04-10  2025      April        2


In [30]:
# Formatting a date back into a specific text format for display, with .dt.strftime()
sales["date_display"] = sales["date"].dt.strftime("%B %d, %Y")   # e.g. "January 05, 2025"
print(sales[["date", "date_display"]].head())


        date       date_display
0 2025-04-22     April 22, 2025
1 2025-04-12     April 12, 2025
2 2025-12-21  December 21, 2025
3 2025-02-11  February 11, 2025
4 2025-04-10     April 10, 2025


**Common `strftime` codes:** `%Y` = 4-digit year, `%m` = 2-digit month, `%d` = 2-digit day, `%B` = full month name, `%A` = full weekday name. These are worth having a reference for — you'll look them up constantly rather than memorize them.


In [31]:
# Filtering using real date comparisons -- only possible because date is an actual datetime now
q1_sales = sales[(sales["date"] >= "2025-01-01") & (sales["date"] <= "2025-03-31")]
print(f"Q1 rows: {len(q1_sales)}")

# Same thing, using the quarter column we just created
q1_sales_v2 = sales[sales["quarter"] == 1]
print(f"Q1 rows (via quarter column): {len(q1_sales_v2)}")


Q1 rows: 995
Q1 rows (via quarter column): 995


---
## Guided Practice

Work through these using `sales`. Start fresh in Exercise 1 to also bring back the missing-value handling from Class 6, since this notebook only re-did dates and duplicates above.


### Exercise 1 — Full cleaning pipeline
Starting from a fresh `pd.read_csv("retail_sales.csv")`: convert `date` to datetime, drop duplicates, fill missing `units_sold` and `revenue` with their column medians, and clean `category` into a new `category_clean` column using the same three steps from today (lowercase+strip, regex normalize separators, fix the `"cloth ing"` typo). Confirm `category_clean` has exactly 5 unique values.


In [ ]:
# Exercise 1 — your code here



### Exercise 2 — Regex practice
Given the list of strings below, use `re.sub()` with a regex pattern to remove all digits from each string, leaving only the letters and spaces.
```python
messy = ["store123", "region 4B", "code99x"]
```


In [ ]:
# Exercise 2 — your code here
messy = ["store123", "region 4B", "code99x"]



### Exercise 3 — Applying `.str.contains()`
Using your cleaned `category_clean` column, filter the dataset to rows where the category contains the letter `"o"` (case-insensitive). Which categories match?


In [ ]:
# Exercise 3 — your code here



### Exercise 4 — Dates
Using the `date` column, create a new column `is_end_of_month` that is `True` if the date falls in the last 3 days of its month, and `False` otherwise.

*Hint: look up `.dt.days_in_month` and `.dt.day`.*


In [ ]:
# Exercise 4 — your code here



### Exercise 5 (stretch) — Save the cleaned dataset
Using your fully cleaned DataFrame from Exercise 1, drop the original messy `category` column, rename `category_clean` to `category`, and save the result as a new file called `retail_sales_clean.csv`. This is the file we'll use starting Class 8.


In [ ]:
# Exercise 5 — your code here



---
## Wrap-up

**Recap:** the `.str` accessor for whole-column text operations; regular expressions for pattern-based matching and cleaning (`re.sub()`, `.str.replace(regex=True)`); `.str.extract()` for pulling structured pieces out of text; exact-value corrections with `.replace()` for typos regex can't catch; and more date operations (`.dt.year`, `.dt.month_name()`, `.dt.strftime()`, date range filtering).

**The big picture:** since Class 4, `retail_sales.csv` has had three real problems — missing values, duplicates, and inconsistent category text. As of today, all three are fixed. Every grouped result from Class 5 onward was quietly wrong because of this; from here on, our analysis is finally trustworthy.

**Common mistakes to watch for:**
- Forgetting `regex=True` in `.str.replace()` when using a regex pattern (without it, pandas treats your pattern as a literal string to search for)
- Using `.replace()` (exact match) when you meant `.str.replace()` (substring/pattern match), or vice versa
- Cleaning `category` but forgetting to actually use the new `category_clean` column afterward — old code still referencing `category` won't benefit from the fix

**HW1 is due before this class. HW2 is assigned today** — it will build on this cleaning pipeline.

**Before next class (Sep 28):** Class 8 starts visualization — matplotlib — now working with genuinely clean data for the first time all semester.
